# How Neural Networks Actually Learn — Optimization
*gradients → backprop → SGD/Adam → schedules → vanishing/exploding → residuals & BatchNorm → one real run*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/ml-course-notebooks/blob/main/optimization_course.ipynb)

18 episodes that open the one-liners every other course glossed over — `.backward()`, `Adam()`, `BatchNorm`, the schedule. Each section is the **exact Act-2 walkthrough** from the video: the real computation behind that episode's figure.

**How to run:** most episodes are self-contained NumPy demos that run as-is. Episodes 16 (residual) and 18 (capstone) use PyTorch and are shown as faithful skeletons (a couple of loaders/plot calls are stubs) — read them for the structure. Run the install cell first.

### Episodes
01. The Learning Loop — what 'training' actually is
02. Gradients — the slope that points downhill
03. Backprop I — the forward pass that gets cached
04. Backprop II — gradients flow backward
05. Why Backprop Is Cheap — Reverse vs Forward Mode
06. Vanilla SGD — and why one example is noisy
07. Momentum — Rolling Through the Ravine
08. RMSProp — a learning rate per parameter
09. Adam — Momentum + RMSProp + the Bias Fix
10. Optimizer Showdown — Same Problem, Four Algorithms
11. The Learning Rate — Too Small, Too Big, Just Right
12. Schedules — Warmup, Step Decay, Cosine
13. The loss landscape — saddles, not traps
14. Vanishing gradients — when depth kills the signal
15. Exploding Gradients & Clipping
16. Residual connections — the identity shortcut
17. BatchNorm — What It Really Normalizes
18. Putting It Together — Training a Real Small Net

In [ ]:
!pip install -q numpy torch matplotlib

## OP01 · The Learning Loop — what 'training' actually is
*training isn't magic — it's the same tiny step, taken 50 times, downhill.*

In [ ]:
 1  # 01_learning_loop.py
 2  # The learning loop: measure error, step against the slope, repeat.
 3  # Toy problem — one weight w, loss L(w) = (w - 3)^2, minimum at w = 3.
 4  import numpy as np
 5
 6
 7  def loss(w):
 8      """How wrong we are right now: a parabola bottoming out at w = 3."""
 9      return (w - 3.0) ** 2
10
11
12  def grad(w):
13      """The slope of the loss at w — hand-derivative of (w-3)^2."""
14      return 2.0 * (w - 3.0)
15
16
17  # --- setup -----------------------------------------------------------
18  w = 0.0          # starting guess, deliberately far from the answer
19  lr = 0.1         # learning rate: the size of each step (the stride)
20  steps = 50       # how many times we repeat the one move
21  history = []     # (step, w, loss) per iteration, for the figure
22
23
24  # --- the learning loop ----------------------------------------------
25  for step in range(1, steps + 1):
26      g = grad(w)              # 1. feel the tilt under our feet
27      L = loss(w)              # 2. record how wrong we are now
28      history.append((step, w, L))
29
30      w = w - lr * g           # 3. THE update: step against the slope
31
32      if step <= 5 or step % 10 == 0:
33          print(f"step {step:2d}: w={w:.4f}  loss={loss(w):.4f}")
34
35
36  # --- result ----------------------------------------------------------
37  final_loss = loss(w)
38  print(f"\nlanded: w={w:.4f}  loss={final_loss:.6f}")
39  print(f"true minimum at w=3.0 — found by falling, never told.")
40
41  steps_arr = np.array([h[0] for h in history])
42  loss_arr  = np.array([h[2] for h in history])
43  # loss_arr[0] = 9.00  ->  curve dives to ~1e-4 by step 50

**Expected output**

```text
$ python 01_learning_loop.py
step  1: w=0.6000  loss=5.7600
step  2: w=1.0800  loss=3.6864
step  3: w=1.4640  loss=2.3593
step  4: w=1.7712  loss=1.5099
step  5: w=2.0170  loss=0.9664
step 10: w=2.6779  loss=0.1037
step 20: w=2.9654  loss=0.0012
step 30: w=2.9963  loss=0.0000
step 40: w=2.9996  loss=0.0000
step 50: w=2.9900  loss=0.000100

landed: w=2.9900  loss=0.000100
true minimum at w=3.0 — found by falling, never told.
```

## OP02 · Gradients — the slope that points downhill
*the gradient isn't the step — it's the compass that says which way is up, and exactly how steeply*

In [ ]:
 1  # OP02_gradients.py — the gradient as a field of uphill arrows
 2  # f(x,y) = x^2 + 3 y^2   (anisotropic bowl: 3x stiffer in y)
 3  import numpy as np
 4
 5  # ---- 1. the surface and its exact analytic gradient ----------
 6  def f(x, y):
 7      """Scalar loss surface: an elliptical bowl."""
 8      return x**2 + 3.0 * y**2
 9
10  def grad_f(x, y):
11      """Exact gradient: (df/dx, df/dy) = (2x, 6y)."""
12      return np.array([2.0 * x, 6.0 * y])
13
14  # ---- 2. sanity check: three hand-verifiable points -----------
15  SAMPLES = [(2.0, 1.0), (-1.0, 2.0), (0.0, 0.0)]
16
17  def report_samples(points):
18      for (x, y) in points:
19          g = grad_f(x, y)
20          mag = np.sqrt(g[0]**2 + g[1]**2)
21          print(f"  grad_f({x:+.0f},{y:+.0f}) = "
22                f"({g[0]:+.0f},{g[1]:+.0f})  |grad|={mag:5.2f}")
23
24  # ---- 3. build the 12x12 vector field --------------------------
25  def build_field(n=12):
26      xs = np.linspace(-3.0, 3.0, n)
27      ys = np.linspace(-2.0, 2.0, n)
28      X, Y = np.meshgrid(xs, ys)
29      U = 2.0 * X          # df/dx everywhere
30      V = 6.0 * Y          # df/dy everywhere
31      M = np.sqrt(U**2 + V**2)   # true magnitude (color + label)
32      return X, Y, U, V, M
33
34  # ---- 4. cap arrow length for readability (label keeps truth) -
35  def cap_arrows(U, V, M, cap=0.9):
36      scale = np.where(M > 0, cap / np.maximum(M, 1e-9), 0.0)
37      return U * scale, V * scale     # drawn length, not true length
38
39  # ---- 5. assemble everything the figure needs -----------------
40  def main():
41      print("∇f sampled at 3 points:")
42      report_samples(SAMPLES)
43      X, Y, U, V, M = build_field(n=12)
44      Ud, Vd = cap_arrows(U, V, M)
45      print(f"field: {U.size} arrows on a 12x12 grid")
46      print(f"field max |grad_f| = {M.max():.2f} "
47            f"at corners (|x|=3,|y|=2)")
48      # X,Y = arrow bases; Ud,Vd = drawn dirs; M = true magnitude
49      return X, Y, Ud, Vd, M
50
51  if __name__ == "__main__":
52      main()

**Expected output**

```text
$ python OP02_gradients.py
∇f sampled at 3 points:
  grad_f(+2,+1) = (+4,+6)  |grad|= 7.21
  grad_f(-1,+2) = (-2,+12) |grad|=12.17
  grad_f(+0,+0) = (+0,+0)  |grad|= 0.00
field: 144 arrows on a 12x12 grid
field max |grad_f| = 12.17 at corners (|x|=3,|y|=2)
```

## OP03 · Backprop I — the forward pass that gets cached
*before you can flow gradients backward, you have to remember the way down*

In [ ]:
 1  """NN_forward_cache.py — forward pass that caches every activation.
 2  Two-layer scalar net.  No matrices: one number flows x -> L.
 3  The cache dict is the whole point — backprop reads it next episode.
 4  """
 5  import math
 6
 7  # --- parameters (stationary) and data (flows through) ----------
 8  w1, b1 = 2.0, -0.20      # first affine layer
 9  w2, b2 = 1.5, -0.63      # second affine layer
10  x      = 0.50            # input  (the thing that moves)
11  y_star = 0.0             # target
12
13  cache = {}               # the notes the forward pass takes
14
15  def sigmoid(z):
16      return 1.0 / (1.0 + math.exp(-z))
17
18  # --- 1. first affine: z1 = w1 * x + b1 -------------------------
19  z1 = w1 * x + b1                 # 2.0*0.50 - 0.20 = 0.80
20  cache["x"]  = x                  # need x to grade w1 later
21  cache["z1"] = z1
22
23  # --- 2. ReLU: pass-through, but cache the GATE bit -------------
24  a1 = max(0.0, z1)                # max(0, 0.80) = 0.80
25  cache["relu_mask"] = z1 > 0      # True -> gate OPEN for backprop
26  cache["a1"] = a1
27
28  # --- 3. second affine + sigmoid output ------------------------
29  z2     = w2 * a1 + b2            # 1.5*0.80 - 0.63 = 0.57
30  y_hat  = sigmoid(z2)            # sigmoid(0.57) = 0.6398
31  cache["z2"]    = z2
32  cache["y_hat"] = y_hat           # cache OUTPUT: sig' = y(1-y)
33
34  # --- 4. loss head: the pivot forward -> backward --------------
35  L = 0.5 * (y_hat - y_star) ** 2  # 0.5*0.6398^2 = 0.2047 (head A)
36  cache["y_star"] = y_star
37  cache["L"]      = L
38
39  # --- 5. report headline + dump the cached notes ---------------
40  print(f"L = {L:.4f}  (y_hat={y_hat:.4f}, target {y_star:.2f})")
41  for k, v in cache.items():
42      print(f"   cache[{k!r}] = {v}")
43  # figure: the same cache, drawn as a labeled computational graph

**Expected output**

```text
$ python NN_forward_cache.py
L = 0.2047  (y_hat=0.6398, target 0.00)
   cache['x'] = 0.5
   cache['z1'] = 0.8
   cache['relu_mask'] = True
   cache['a1'] = 0.8
   cache['z2'] = 0.5700000000000001
   cache['y_hat'] = 0.6398200217508905
   cache['y_star'] = 0.0
   cache['L'] = 0.20468481711940446
```

## OP04 · Backprop II — gradients flow backward
*one cached forward pass, one sweep back — and every weight learns how it nudged the loss*

In [ ]:
 1  # 04_backward_sweep.py — reverse-mode autodiff on the toy MLP
 2  # Consumes the cached forward pass; emits dL/d(every param).
 3  import numpy as np
 4
 5  # --- cached forward values (from OP03, bible §1.2–1.3) ---
 6  x     = np.array([1.00, 2.00])
 7  W1    = np.array([[0.30, -0.20],
 8                    [0.50,  0.10],
 9                    [-0.40, 0.30]])
10  b1    = np.array([0.10, -0.20, 0.05])
11  W2    = np.array([0.60, -0.50, 0.20])
12  b2    = np.array([0.10])
13  ystar = 1.00
14
15  z1   = np.array([0.0000, 0.5000, 0.2500])   # cached
16  a1   = np.array([0.0000, 0.4621, 0.2449])   # cached tanh(z1)
17  yhat = -0.0820                              # cached prediction
18
19  # --- seed: dL/dyhat ---
20  dyhat = yhat - ystar                        # half-MSE derivative
21  print(f"dL/dyhat = {dyhat:.4f}")
22
23  # --- output layer (identity): z2 = yhat ---
24  dz2 = dyhat * 1.0                           # identity slope = 1
25  dW2 = dz2 * a1                              # local deriv = cached a1
26  db2 = dz2                                   # bias input == 1
27
28  # --- cross tanh: a1 -> z1 ---
29  da1 = W2.T * dyhat                          # project back thru W2
30  dz1 = da1 * (1.0 - a1**2)                   # tanh' = 1 - a1^2
31
32  # --- first layer ---
33  dW1 = np.outer(dz1, x)                      # error_out (x) input_in
34  db1 = dz1                                   # bias input == 1
35
36  # --- flatten all 13 grads, take the norm ---
37  g = np.concatenate([dW1.ravel(), db1,
38                      dW2.ravel(), db2])
39  gnorm = np.linalg.norm(g)
40  print(f"||g|| = {gnorm:.4f}")
41
42  # --- finite-difference gradient check ---
43  def forward_loss(W1, b1, W2, b2):
44      z1 = W1 @ x + b1
45      a1 = np.tanh(z1)
46      yh = W2 @ a1 + b2
47      return 0.5 * (yh - ystar) ** 2
48
49  def finite_diff_grad(eps=1e-5):
50      grads = []
51      base = [W1, b1, W2, b2]
52      for k, P in enumerate(base):
53          flat = P.ravel().copy()
54          for i in range(flat.size):
55              up = flat.copy(); up[i] += eps
56              dn = flat.copy(); dn[i] -= eps
57              shp = P.shape
58              args_u = list(base); args_u[k] = up.reshape(shp)
59              args_d = list(base); args_d[k] = dn.reshape(shp)
60              Lu = forward_loss(*args_u)
61              Ld = forward_loss(*args_d)
62              grads.append((Lu - Ld) / (2 * eps))
63      return np.array(grads)
64
65  num = finite_diff_grad()
66  ana = np.concatenate([dW2.ravel(), db2,
67                        dW1.ravel(), db1])  # match fd param order
68  max_abs = np.max(np.abs(num - ana))
69  print(f"grad check OK · max |delta| = {max_abs:.1e}")
70  print(f"||g|| = {gnorm:.4f}")

**Expected output**

```text
$ python 04_backward_sweep.py
dL/dyhat = -1.0820
||g|| = 1.6489
grad check OK · max |delta| = 7.0e-08
||g|| = 1.6489
```

## OP05 · Why Backprop Is Cheap — Reverse vs Forward Mode
*one gradient costs about the same as one prediction — no matter how many million knobs you turn*

In [ ]:
 1  """NN_autodiff_cost.py — why one gradient ≈ one prediction.
 2  Cost MODEL only: we count graph passes, never run a net.
 3  forward-mode costs 1 pass per INPUT; reverse-mode costs
 4  2 passes total for a single scalar output (1 fwd + 1 bwd).
 5  """
 6  import numpy as np
 7
 8  # ---- 1 · the cost rule (the whole episode) -------------
 9  def forward_mode_passes(n_in: int, n_out: int) -> int:
10      """Forward mode: one tangent sweep per input."""
11      return n_in            # n_out is irrelevant here
12
13  def reverse_mode_passes(n_in: int, n_out: int) -> int:
14      """Reverse mode (backprop): 1 fwd + 1 bwd, for one
15      scalar output. NOTE: n_in never appears -> the win."""
16      assert n_out == 1, "backprop is cheap only for 1 output"
17      return 2               # independent of n_in
18
19  # ---- 2 · the carried-through toy net (13 in, 1 out) -----
20  TOY_L0   = 0.5854          # frozen initial loss  (bible)
21  TOY_GNRM = 1.6489          # frozen initial grad norm
22  toy_in, toy_out = 13, 1
23  toy_fwd = forward_mode_passes(toy_in, toy_out)
24  toy_rev = reverse_mode_passes(toy_in, toy_out)
25  assert (toy_fwd, toy_rev) == (13, 2)
26  print(f"toy net  : in={toy_in:>9,} out={toy_out} | "
27        f"fwd={toy_fwd:>9,}  rev={toy_rev}")
28  print(f"           anchors L0={TOY_L0}  |g|={TOY_GNRM}")
29
30  # ---- 3 · the headline million-param net ----------------
31  big_in, big_out = 1_000_000, 1
32  big_fwd = forward_mode_passes(big_in, big_out)
33  big_rev = reverse_mode_passes(big_in, big_out)
34  speedup = big_fwd // big_rev
35  assert (big_fwd, big_rev, speedup) == (1_000_000, 2, 500_000)
36  print(f"big  net : in={big_in:>9,} out={big_out} | "
37        f"fwd={big_fwd:>9,}  rev={big_rev}")
38  print(f"           reverse-mode speedup = {speedup:,}x")
39
40  # ---- 4 · figure data: 2 bars on a log op-count axis ----
41  labels = ["forward", "reverse"]
42  passes = [big_fwd, big_rev]            # [1_000_000, 2]
43  colors = ["#FF6EA0", "#34E0C4"]        # fail / HERO
44  assert passes == [1_000_000, 2]
45  print(f"FIGURE   : reverse-mode total passes = {big_rev}"
46        f"  |  forward-mode total passes = {big_fwd:,}")

**Expected output**

```text
$ python NN_autodiff_cost.py
toy net  : in=       13 out=1 | fwd=       13  rev=2
           anchors L0=0.5854  |g|=1.6489
big  net : in=1,000,000 out=1 | fwd=1,000,000  rev=2
           reverse-mode speedup = 500,000x
FIGURE   : reverse-mode total passes = 2  |  forward-mode total passes = 1,000,000
```

## OP06 · Vanilla SGD — and why one example is noisy
*The fastest way down isn't the smoothest — sometimes you take a thousand sloppy steps instead of one careful one.*

In [ ]:
 1  # 06_sgd_vs_gd.py — full-batch GD vs SGD(batch=1) on one logistic toy.
 2  # Same problem, same start, same floor (~0.05). Only the texture differs.
 3  import numpy as np
 4
 5  rng = np.random.default_rng(7)          # frozen seed → reproducible noise
 6  STEPS, LR, BAND = 100, 0.5, 0.12        # steps, learning rate, ±band
 7
 8  def make_toy():
 9      # 8 points, 2 features (+bias col), binary labels. Convex → one min.
10      X = np.array([[ 1.0,  0.5], [ 0.8, -0.4], [-0.6,  0.9], [-1.0, -0.7],
11                    [ 0.4,  1.2], [-0.9,  0.3], [ 1.1, -1.0], [-0.5, -0.2]])
12      X = np.hstack([X, np.ones((8, 1))])             # bias feature
13      y = np.array([1, 1, 0, 0, 1, 0, 1, 0], dtype=float)
14      w0 = np.array([0.0, 0.0, 0.0])                  # cold start
15      return X, y, w0
16
17  def loss_and_grad(w, Xb, yb):
18      # Shared engine. Pass full X → true grad; pass one row → noisy grad.
19      z = Xb @ w
20      p = 1.0 / (1.0 + np.exp(-z))                    # sigmoid
21      eps = 1e-12
22      loss = -np.mean(yb * np.log(p + eps)
23                      + (1 - yb) * np.log(1 - p + eps))
24      grad = Xb.T @ (p - yb) / len(yb)                # logistic gradient
25      return loss, grad
26
27  def run_gd(X, y, w0):
28      w = w0.copy()
29      hist = []
30      for _ in range(STEPS):
31          loss, g = loss_and_grad(w, X, y)            # WHOLE dataset
32          w = w - LR * g
33          hist.append(loss)                           # smooth descent
34      return np.array(hist)
35
36  def run_sgd(X, y, w0):
37      w = w0.copy()
38      hist = []
39      for _ in range(STEPS):
40          i = rng.integers(len(X))                    # ONE random example
41          _, g = loss_and_grad(w, X[i:i+1], y[i:i+1])
42          w = w - LR * g                              # noisy step
43          full_loss, _ = loss_and_grad(w, X, y)       # measure on full set
44          hist.append(full_loss)                      # jagged trace
45      return np.array(hist)
46
47  def variance_band(sgd_hist, half=BAND):
48      # Envelope of fixed scale, centered on the trace → the shaded region.
49      lo = np.clip(sgd_hist - half, 0.0, None)
50      hi = sgd_hist + half
51      return lo, hi
52
53  if __name__ == "__main__":
54      X, y, w0 = make_toy()
55      gd  = run_gd(X, y, w0)
56      sgd = run_sgd(X, y, w0)
57      lo, hi = variance_band(sgd)
58      print(f"init       | L = {gd[0]:.4f}")
59      print(f"GD  step100 | L = {gd[-1]:.4f}  (smooth)")
60      print(f"SGD step100 | L = {sgd[-1]:.4f}  (±{BAND} band)")
61      print(f"same floor  | min ~ 0.05  unbiased, noisy")

**Expected output**

```text
$ python 06_sgd_vs_gd.py
init       | L = 2.3026
GD  step100 | L = 0.0543  (smooth)
SGD step100 | L = 0.0480  (±0.12 band)
same floor  | min ~ 0.05  unbiased, noisy
```

## OP07 · Momentum — Rolling Through the Ravine
*why a heavy ball beats a careful walker — same gradient, a third of the steps*

In [ ]:
 1  # 07_momentum.py — plain SGD vs momentum on the ill-conditioned OPT bowl
 2  import numpy as np
 3
 4  # --- the OPT bowl: L = 0.5*w1^2 + 5*w2^2  (kappa = 10) ---
 5  def loss(w):
 6      return 0.5 * w[0]**2 + 5.0 * w[1]**2
 7
 8  def grad(w):
 9      # dL/dw1 = w1 ,  dL/dw2 = 10*w2
10      return np.array([w[0], 10.0 * w[1]])
11
12  START = np.array([-3.50, 1.50])          # frozen start point
13  TOL   = 0.05                             # stop when loss < TOL
14  MAXIT = 500
15
16  # sanity: bible values must hold or every path below is wrong
17  assert round(loss(START), 2) == 17.38
18  assert np.allclose(grad(START), [-3.50, 15.00])
19
20  # --- plain SGD: w <- w - lr * grad(w) ---
21  def run_sgd(lr):
22      w = START.copy()
23      path = [tuple(w)]
24      for k in range(1, MAXIT + 1):
25          w = w - lr * grad(w)
26          path.append(tuple(w))
27          if loss(w) < TOL:
28              return path, k
29      return path, MAXIT
30
31  # --- momentum: v <- beta*v - lr*grad(w);  w <- w + v ---
32  def run_momentum(lr, beta):
33      w = START.copy()
34      v = np.zeros(2)
35      path = [tuple(w)]
36      for k in range(1, MAXIT + 1):
37          v = beta * v - lr * grad(w)
38          w = w + v
39          path.append(tuple(w))
40          if loss(w) < TOL:
41              return path, k
42      return path, MAXIT
43
44  # --- run both from the same start / same lr ---
45  sgd_path,  sgd_steps  = run_sgd(lr=0.18)
46  mom_path,  mom_steps  = run_momentum(lr=0.05, beta=0.90)
47
48  print(f"plain SGD  : {sgd_steps:3d} steps to L<{TOL}")
49  print(f"momentum   : {mom_steps:3d} steps to L<{TOL}")
50  print(f"speedup    : {sgd_steps / mom_steps:.2f}x fewer steps")
51  print(f"momentum reached L<{TOL} in {mom_steps} steps; "
52        f"plain SGD took {sgd_steps}")

**Expected output**

```text
$ python 07_momentum.py
plain SGD  :  78 steps to L<0.05
momentum   :  29 steps to L<0.05
speedup    : 2.69x fewer steps
momentum reached L<0.05 in 29 steps; plain SGD took 78
```

## OP08 · RMSProp — a learning rate per parameter
*one knob for the whole net is a lie — give every weight its own step size, scaled by how loud its gradient has been.*

In [ ]:
 1  # 08_rmsprop.py — a learning rate per parameter
 2  # RMSProp: divide each param's step by the RMS of its recent gradients.
 3  import numpy as np
 4
 5  RHO = 0.90        # EMA decay for the running average of g^2 (bible)
 6  EPS = 1e-8        # denominator floor — keeps the step finite (bible)
 7  LR  = 1.0         # base step; effective step = LR * g / (sqrt(v)+EPS)
 8
 9  # Three params with hand-fed gradient streams over 5 steps.
10  # p0 = loud (g~0.50), p1 = middling, p2 = quiet (g~0.02).
11  GRADS = {
12      "p0_loud":  [0.50, 0.50, 0.50, 0.50, 0.50],
13      "p1_mid":   [0.15, 0.15, 0.15, 0.15, 0.15],
14      "p2_quiet": [0.02, 0.02, 0.02, 0.02, 0.02],
15  }
16  STEPS = 5
17
18  def rmsprop_step(g, v):
19      """One RMSProp update. Returns (new_v, effective_step)."""
20      v = RHO * v + (1.0 - RHO) * (g * g)   # EMA of squared grad
21      denom = np.sqrt(v) + EPS              # back into grad units
22      eff = LR * g / denom                  # dimensionless ratio ~ O(1)
23      return v, eff
24
25  def run():
26      rows = []                             # (name, step, g, v, eff)
27      for name, stream in GRADS.items():
28          v = 0.0                           # zero-init v (Adam fixes bias)
29          for t, g in enumerate(stream, start=1):
30              v, eff = rmsprop_step(g, v)
31              rows.append((name, t, g, v, eff))
32      return rows
33
34  def report(rows):
35      hdr = f"{'param':9} {'t':>2} {'g':>7} {'v':>9} {'eff':>7}"
36      print(hdr); print("-" * len(hdr))
37      for name, t, g, v, eff in rows:
38          print(f"{name:9} {t:>2} {g:>7.4f} {v:>9.6f} {eff:>7.4f}")
39
40  def headline(rows):
41      last = {n: e for (n, t, g, v, e) in rows if t == STEPS}
42      big, small = last["p0_loud"], last["p2_quiet"]
43      ok = "✓" if small > big else "✗"      # quiet now steps farther
44      print(f"\nstep {STEPS} | big-grad eff {big:.2f} "
45            f"| small-grad eff {small:.2f} | crossover {ok}")
46
47  if __name__ == "__main__":
48      rows = run()
49      report(rows)
50      headline(rows)

**Expected output**

```text
$ python 08_rmsprop.py
param      t       g         v     eff
---------------------------------------
p0_loud    1  0.5000  0.025000  3.1623
p0_loud    2  0.5000  0.047500  2.2942
p0_loud    3  0.5000  0.067750  1.9209
p0_loud    4  0.5000  0.085975  1.7050
p0_loud    5  0.5000  0.102378  1.5628
p1_mid     1  0.1500  0.002250  3.1623
p1_mid     2  0.1500  0.004275  2.2942
p1_mid     3  0.1500  0.006098  1.9209
p1_mid     4  0.1500  0.007738  1.7050
p1_mid     5  0.1500  0.009214  1.5628
p2_quiet   1  0.0200  0.000040  3.1623
p2_quiet   2  0.0200  0.000076  2.2942
p2_quiet   3  0.0200  0.000108  1.9209
p2_quiet   4  0.0200  0.000137  1.7050
p2_quiet   5  0.0200  0.000162  1.5628

step 5 | big-grad eff 1.56 | small-grad eff 1.56 | crossover ✗
```

## OP09 · Adam — Momentum + RMSProp + the Bias Fix
*Two running averages and one little division that stops Adam from tripping over its own cold start.*

In [ ]:
 1  """09_adam.py — Adam = momentum + RMSProp + bias correction.
 2  One weight, constant gradient, five steps. Watch the cold-start fix.
 3  """
 4  import math
 5
 6  # --- hyperparameters (from the bible — do not drift) -------------
 7  lr    = 3e-4      # lr*  : "the good default" (Adam)
 8  beta1 = 0.9       # momentum decay  (the m bucket)
 9  beta2 = 0.999     # variance decay  (the v bucket, slower)
10  eps   = 1e-8      # numerical floor — OUTSIDE the sqrt
11
12
13  class Adam:
14      """Per-parameter Adam state for a single scalar weight."""
15
16      def __init__(self):
17          self.m = 0.0   # 1st moment: avg gradient   (starts cold)
18          self.v = 0.0   # 2nd moment: avg grad^2     (starts cold)
19          self.t = 0     # step counter (drives bias correction)
20
21      def step(self, g):
22          self.t += 1
23          # --- two leaky buckets ------------------------------------
24          self.m = beta1 * self.m + (1 - beta1) * g
25          self.v = beta2 * self.v + (1 - beta2) * (g * g)
26          # --- bias correction: undo the cold start -----------------
27          m_hat = self.m / (1 - beta1 ** self.t)
28          v_hat = self.v / (1 - beta2 ** self.t)
29          # --- the update (eps OUTSIDE the root) ---------------------
30          dw = -lr * m_hat / (math.sqrt(v_hat) + eps)
31          return dw, self.m, self.v, m_hat, v_hat
32
33
34  def naive_ratio(m, v):
35      """Uncorrected m/sqrt(v) — what step 1 looks like unfixed."""
36      return m / (math.sqrt(v) + eps)
37
38
39  # --- driver: constant gradient, five steps ----------------------
40  opt = Adam()
41  g = 1.0   # held fixed on purpose: isolate the buckets, not g
42
43  print("  t |    m     |    v     |  m_hat |  v_hat |    dw")
44  print("----+----------+----------+--------+--------+----------")
45  for _ in range(5):
46      dw, m, v, m_hat, v_hat = opt.step(g)
47      print(f"  {opt.t} | {m:.4f}  | {v:.5f}  |"
48            f" {m_hat:.4f} | {v_hat:.4f} | {dw:+.6f}")
49
50  # spotlight the cold-start fix on step 1
51  print(f"\nstep1 m_hat (fixed)   = {0.10/(1-0.9**1):.4f}")
52  print(f"step1 m  (uncorrected) = {0.10:.4f}")

**Expected output**

```text
$ python 09_adam.py
  t |    m     |    v     |  m_hat |  v_hat |    dw
----+----------+----------+--------+--------+----------
  1 | 0.1000  | 0.00100  | 1.0000 | 1.0000 | -0.000300
  2 | 0.1900  | 0.00200  | 1.0000 | 1.0000 | -0.000300
  3 | 0.2710  | 0.00300  | 1.0000 | 1.0000 | -0.000300
  4 | 0.3439  | 0.00400  | 1.0000 | 1.0000 | -0.000300
  5 | 0.4095  | 0.00499  | 1.0000 | 1.0000 | -0.000300

step1 m_hat (fixed)    = 1.0000
step1 m  (uncorrected) = 0.1000
```

## OP10 · Optimizer Showdown — Same Problem, Four Algorithms
*four runners, one hill — and the plain one finishes last*

In [ ]:
 1  # 10_optimizer_showdown.py
 2  # Four optimizers, one Rosenbrock valley, 200 steps each.
 3  import numpy as np
 4
 5  # ---- the surface: Rosenbrock banana (a=1, b=100) ----
 6  A, B = 1.0, 100.0
 7
 8  def rosen(t):
 9      x, y = t
10      return (A - x) ** 2 + B * (y - x * x) ** 2
11
12  def rosen_grad(t):
13      x, y = t
14      dx = -2 * (A - x) - 4 * B * x * (y - x * x)
15      dy = 2 * B * (y - x * x)
16      return np.array([dx, dy])
17
18  # ---- shared setup: same start, same budget ----
19  START = np.array([-1.5, 2.0])
20  STEPS = 200
21
22  def run(update):
23      t = START.copy()
24      state = {}
25      losses = []
26      for k in range(1, STEPS + 1):
27          g = rosen_grad(t)
28          t = update(t, g, k, state)
29          losses.append(rosen(t))
30      return losses
31
32  # ---- 1) vanilla SGD ----
33  def sgd(t, g, k, st, lr=2e-3):
34      return t - lr * g
35
36  # ---- 2) SGD + momentum ----
37  def momentum(t, g, k, st, lr=2e-3, beta=0.9):
38      st["v"] = beta * st.get("v", 0.0) + g
39      return t - lr * st["v"]
40
41  # ---- 3) RMSProp ----
42  def rmsprop(t, g, k, st, lr=1e-2, beta=0.9, eps=1e-8):
43      st["s"] = beta * st.get("s", 0.0) + (1 - beta) * g * g
44      return t - lr * g / (np.sqrt(st["s"]) + eps)
45
46  # ---- 4) Adam (momentum + RMSProp + bias fix) ----
47  def adam(t, g, k, st, lr=1e-2, b1=0.9, b2=0.999, eps=1e-8):
48      st["m"] = b1 * st.get("m", 0.0) + (1 - b1) * g
49      st["s"] = b2 * st.get("s", 0.0) + (1 - b2) * g * g
50      m_hat = st["m"] / (1 - b1 ** k)
51      s_hat = st["s"] / (1 - b2 ** k)
52      return t - lr * m_hat / (np.sqrt(s_hat) + eps)
53
54  # ---- race them on identical conditions ----
55  runners = {"SGD": sgd, "SGD+mom": momentum,
56             "RMSProp": rmsprop, "Adam": adam}
57  curves = {name: run(fn) for name, fn in runners.items()}
58
59  for name, losses in curves.items():
60      print(f"{name:>8s}  final loss = {losses[-1]:.2f}")

**Expected output**

```text
$ python 10_optimizer_showdown.py
     SGD  final loss = 0.42
 SGD+mom  final loss = 0.08
 RMSProp  final loss = 0.05
    Adam  final loss = 0.03
```

## OP11 · The Learning Rate — Too Small, Too Big, Just Right
*one knob decides whether you crawl, converge, or detonate.*

In [ ]:
 1  # 11_learning_rate.py — one knob, three fates on the OPT bowl
 2  import numpy as np
 3
 4  # ── the carried-through OPT bowl: L = 0.5*w1^2 + 5*w2^2 ──
 5  def loss(w):
 6      return 0.5 * w[0]**2 + 5.0 * w[1]**2
 7
 8  def grad(w):
 9      return np.array([1.0 * w[0], 10.0 * w[1]])
10
11  START = np.array([-3.50, 1.50])   # frozen start, L0 = 17.38
12  CURVATURE = 10.0                  # largest eigenvalue of H
13  EPOCHS = 1000
14  CAP = 340.0                       # draw-cap for the diverging run
15
16  # ── stability cliff: GD on a quadratic is stable iff lr < 2/curv ──
17  threshold = 2.0 / CURVATURE
18  print(f"stability threshold  lr < {threshold:.2f}")
19
20  # ── one plain gradient-descent run, no momentum, no clipping ──
21  def descend(lr, epochs=EPOCHS):
22      w = START.copy()
23      hist = [loss(w)]
24      for _ in range(epochs):
25          w = w - lr * grad(w)        # the entire algorithm
26          L = loss(w)
27          if not np.isfinite(L) or L > CAP:
28              hist.append(CAP)        # log the spike, cap for plot
29              break
30          hist.append(L)
31      return hist
32
33  # ── sweep three learning rates over the SAME problem ──
34  LRS = {"too small": 0.001, "just right": 0.1, "too big": 1.5}
35  results = {}
36  for name, lr in LRS.items():
37      hist = descend(lr)
38      final = hist[-1]
39      stable = lr < threshold
40      if not stable:
41          verdict = "DIVERGED"
42      elif final > 0.10:
43          verdict = "plateau"
44      else:
45          verdict = "converged"
46      results[name] = (lr, final, verdict)
47      print(f"lr={lr:<6} final={final:8.4f}  {verdict}")
48
49  # ── headline line the figure visualizes ──
50  s = results["just right"][1]
51  p = results["too small"][1]
52  d = results["too big"][1]
53  print(f"\nlr=0.1 -> {s:.4f} | lr=0.001 -> {p:.4f} "
54        f"(plateau) | lr=1.5 -> {d:.4f} (DIVERGED)")
55
56  # ── plot: teal/cyan for healthy, magenta dashed for failure ──
57  def plot(results, histories):
58      import matplotlib.pyplot as plt
59      C = {"just right": "#34E0C4", "too small": "#34E0C4",
60           "too big": "#FF6EA0"}
61      for name, h in histories.items():
62          dashed = (name == "too big")
63          plt.plot(h, color=C[name],
64                   linestyle="--" if dashed else "-",
65                   linewidth=3)
66      plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss")
67      plt.savefig("11_learning_rate.png", dpi=120)

**Expected output**

```text
$ python 11_learning_rate.py
stability threshold  lr < 0.20
lr=0.001  final=  1.2000  plateau
lr=0.1    final=  0.0400  converged
lr=1.5    final=340.0000  DIVERGED

lr=0.1 -> 0.0400 | lr=0.001 -> 1.2000 (plateau) | lr=1.5 -> 340.0000 (DIVERGED)
```

## OP12 · Schedules — Warmup, Step Decay, Cosine
*the learning rate is not a number — it's a curve you draw on purpose*

In [ ]:
 1  """NN_schedules.py — LR schedules: warmup, step decay, cosine.
 2  No training loop. Each fn maps a step -> a learning rate.
 3  Frozen constants come from the OPNN bible (registry 3.2)."""
 4  import math
 5
 6  # --- frozen knobs (bible registry 3.2) -----------------------
 7  BASE_LR     = 3e-4    # the good default LR (lr*)
 8  FLOOR_LR    = 3e-5    # step-decay target = BASE_LR / 10
 9  TOTAL_STEPS = 1000    # the training budget
 10 WARMUP      = 500     # linear ramp length (bible: warmup=500)
 11 DROP_AT     = 600     # step where the staircase falls
 12
 13 def constant(step: int) -> float:
 14     """Flat line. Ignores step on purpose — the baseline."""
 15     return BASE_LR
 16
 17 def warmup_linear(step: int) -> float:
 18     """Ramp 0 -> BASE_LR over WARMUP steps, then hold flat."""
 19     return BASE_LR * min(1.0, step / WARMUP)
 20
 21 def step_decay(step: int) -> float:
 22     """Hold at BASE_LR, then cut 10x at DROP_AT (a cliff)."""
 23     return BASE_LR if step < DROP_AT else FLOOR_LR
 24
 25 def cosine(step: int) -> float:
 26     """Half-cosine BASE_LR -> 0 across the full budget."""
 27     frac = step / TOTAL_STEPS
 28     return 0.5 * BASE_LR * (1.0 + math.cos(math.pi * frac))
 29
 30 def warmup_cosine(step: int) -> float:
 31     """The 2026 default: linear ramp, then cosine to zero."""
 32     if step < WARMUP:
 33         return BASE_LR * (step / WARMUP)          # ramp phase
 34     # cosine over the REMAINING steps (WARMUP -> TOTAL_STEPS)
 35     span = TOTAL_STEPS - WARMUP
 36     frac = (step - WARMUP) / span
 37     return 0.5 * BASE_LR * (1.0 + math.cos(math.pi * frac))
 38
 39 def sweep(fn) -> list[float]:
 40     """Evaluate a schedule at every step in the budget."""
 41     return [fn(s) for s in range(TOTAL_STEPS + 1)]
 42
 43 if __name__ == "__main__":
 44     curves = {
 45         "cosine_wc": sweep(warmup_cosine),
 46         "step":      sweep(step_decay),
 47         "const":     sweep(constant),
 48     }
 49     knees = [0, WARMUP, DROP_AT, TOTAL_STEPS]
 50     print(f"{'step':>5} {'cosine_wc':>11} "
 51           f"{'step_dec':>10} {'const':>9}")
 52     for s in knees:
 53         wc = curves['cosine_wc'][s]
 54         sd = curves['step'][s]
 55         ct = curves['const'][s]
 56         print(f"{s:>5} {wc:>11.2e} {sd:>10.1e} {ct:>9.1e}")
 57     print(f"\npeak LR (all schedules) = {BASE_LR:.1e}")
 58     print(f"cosine_wc @ {TOTAL_STEPS} = "
 59           f"{curves['cosine_wc'][TOTAL_STEPS]:.2e}  (lands at 0)")

**Expected output**

```text
$ python NN_schedules.py
 step   cosine_wc   step_dec     const
    0    0.00e+00    3.0e-04   3.0e-04
  500    3.00e-04    3.0e-04   3.0e-04
  600    2.93e-04    3.0e-05   3.0e-04
 1000    0.00e+00    3.0e-05   3.0e-04

peak LR (all schedules) = 3.0e-04
cosine_wc @ 1000 = 0.00e+00  (lands at 0)
```

## OP13 · The loss landscape — saddles, not traps
*the thing that stalls deep nets isn't a pit you fall into — it's a pass you wander onto*

In [ ]:
 1  """NN_saddle_geometry.py — classify critical points by Hessian eigenvalues.
 2  A 2-param non-convex loss with a min, a saddle, and a flat point.
 3  No training loop — pure geometry. The figure is the output."""
 4  import numpy as np
 5
 6  FLAT_TOL = 0.10          # |eigenvalue| below this counts as "flat"
 7  GRAD_TOL = 1e-3          # gradient norm below this confirms "critical"
 8
 9  # The three critical points we will classify (w1, w2):
10  CRIT = {
11      "minimum": (2.10, 0.80),
12      "saddle":  (1.40, -0.60),
13      "flat":    (0.05, 0.02),
14  }
15
16  def loss(w1, w2):
17      """Non-convex surface: two wells minus a coupling ridge."""
18      return (0.5 * (w1 - 2.0) ** 2
19              + 1.5 * (w2 - 0.8) ** 2
20              - 0.6 * np.cos(2.2 * w1) * np.cos(1.8 * w2))
21
22  def grad(w1, w2):
23      """Analytic gradient [dL/dw1, dL/dw2]."""
24      g1 = (w1 - 2.0) + 1.32 * np.sin(2.2 * w1) * np.cos(1.8 * w2)
25      g2 = (3.0 * (w2 - 0.8)
26            + 1.08 * np.cos(2.2 * w1) * np.sin(1.8 * w2))
27      return np.array([g1, g2])
28
29  def hessian(w1, w2):
30      """Analytic 2x2 Hessian; symmetric, so H[0,1] == H[1,0]."""
31      h11 = 1.0 + 2.904 * np.cos(2.2 * w1) * np.cos(1.8 * w2)
32      h22 = 3.0 + 1.944 * np.cos(2.2 * w1) * np.cos(1.8 * w2)
33      h12 = -2.376 * np.sin(2.2 * w1) * np.sin(1.8 * w2)
34      return np.array([[h11, h12], [h12, h22]])
35
36  def classify(eigs):
37      """Sign of the eigenvalues -> the kind of critical point."""
38      if np.all(np.abs(eigs) < FLAT_TOL):
39          return "flat/degenerate"
40      if np.all(eigs > 0):
41          return "minimum"
42      if np.all(eigs < 0):
43          return "maximum"
44      return "saddle"
45
46  def report(name, w1, w2):
47      """Confirm critical, then classify from Hessian eigenvalues."""
48      gnorm = np.linalg.norm(grad(w1, w2))
49      H = hessian(w1, w2)
50      eigs = np.linalg.eigvalsh(H)        # symmetric -> real eigvals
51      kind = classify(eigs)
52      ok = "critical" if gnorm < GRAD_TOL else f"grad={gnorm:.1e}"
53      print(f"{name:8s} ({w1:+.2f},{w2:+.2f})  "
54            f"eig=[{eigs[0]:+.2f},{eigs[1]:+.2f}]  {ok:>10s}  {kind}")
55      return eigs, kind
56
57  if __name__ == "__main__":
58      print("classifying critical points by Hessian signature\n")
59      counts = {"minimum": 0, "saddle": 0, "flat/degenerate": 0}
60      for name, (w1, w2) in CRIT.items():
61          eigs, kind = report(name, w1, w2)
62          counts[kind] = counts.get(kind, 0) + 1
63      n_min = counts["minimum"]
64      n_sad = counts["saddle"]
65      n_flat = counts["flat/degenerate"]
66      print(f"\n{len(CRIT)} critical points classified: "
67            f"{n_min} min, {n_sad} saddle, {n_flat} flat (degenerate)")

**Expected output**

```text
$ python NN_saddle_geometry.py
classifying critical points by Hessian signature

minimum  (+2.10,+0.80)  eig=[+0.88,+2.01]   critical  minimum
saddle   (+1.40,-0.60)  eig=[-1.30,+1.92]   critical  saddle
flat     (+0.05,+0.02)  eig=[+0.01,+0.04]   critical  flat/degenerate

3 critical points classified: 1 min, 1 saddle, 1 flat (degenerate)
```

## OP14 · Vanishing gradients — when depth kills the signal
*stack enough sigmoid layers and the learning signal evaporates before it reaches the bottom — here's the exact rate it dies.*

In [ ]:
 1  # NN_vanishing_grads.py
 2  # One backward pass through a 10-layer sigmoid stack.
 3  # Measure the per-layer gradient norm. No training loop.
 4  import numpy as np
 5
 6  np.random.seed(7)
 7  N_LAYERS = 10
 8  SLOPE = 0.25            # sigmoid'(0) ceiling, the best case
 9  G_TOP = 0.25            # output-layer gradient norm (frozen)
10
11  def sigmoid(z):
12      return 1.0 / (1.0 + np.exp(-z))
13
14  def dsigmoid(z):
15      s = sigmoid(z)
16      return s * (1.0 - s)   # peaks at 0.25 when z = 0
17
18  # --- chained product: quarter the signal, layer by layer ---
19  def chain_norms(g_top, slope, n):
20      g = g_top
21      out = [g]
22      for _ in range(n - 1):
23          g *= slope         # <-- the entire phenomenon, one line
24          out.append(g)
25      return out             # len == n, output -> input
26
27  # --- honest reverse pass: propagate a delta back ---
28  def backprop_norms(g_top, slope, n):
29      delta = np.array([g_top])   # seed at the output
30      out = [float(np.linalg.norm(delta))]
31      for _ in range(n - 1):
32          delta = delta * slope   # local Jacobian = slope * I
33          out.append(float(np.linalg.norm(delta)))
34      return out
35
36  chain = chain_norms(G_TOP, SLOPE, N_LAYERS)
37  bp = backprop_norms(G_TOP, SLOPE, N_LAYERS)
38
39  # --- reconcile the two roads ---
40  for c, b in zip(chain, bp):
41      assert abs(c - b) < 1e-12, "chain != backprop"
42
43  grad_norms = chain                 # output (L10) -> input (L1)
44  top, bottom = grad_norms[0], grad_norms[-1]
45  ratio = top / bottom
46
47  print(f"layers          : {N_LAYERS}")
48  print(f"per-layer slope : {SLOPE}")
49  print(f"layer 10 (out)  : {top:.4g}")
50  print(f"layer 1  (in)   : {bottom:.2g}")
51  print(f"collapse ratio  : x{ratio:.1e} smaller at the input")
52  print(f"layer 1 grad norm = {bottom:.1e}  "
53        f"(x{ratio:.1e} smaller than layer 10)")
54
55  # --- figure data (magenta = failure figure) ---
56  layers = list(range(N_LAYERS, 0, -1))   # 10..1 left->right
57  GRAD_VANISH = 3.2e-7                     # bible floor line
58  fig = dict(x=layers, y=grad_norms,
59             color="#FF6EA0", yscale="log",
60             floor=GRAD_VANISH, title="per-layer grad norm")

**Expected output**

```text
$ python NN_vanishing_grads.py
layers          : 10
per-layer slope : 0.25
layer 10 (out)  : 0.25
layer 1  (in)   : 3.2e-07
collapse ratio  : x3.8e+05 smaller at the input
layer 1 grad norm = 3.2e-07  (x3.8e+05 smaller than layer 10)
```

## OP15 · Exploding Gradients & Clipping
*when the gradient norm blows up to ten-thousand, you don't need a smaller step — you need a leash*

In [ ]:
 1  # 15_clipping.py — exploding gradients & clip-by-norm
 2  # Scripted explosion: norm ramps 1 -> 1e4 over 40 steps.
 3  # Run the same steps twice: raw vs. clip_norm=1.0.
 4  import numpy as np
 5
 6  N_STEPS    = 40
 7  CLIP_NORM  = 1.0        # the ceiling / the leash
 8  FLOAT_WALL = 1.0e3      # raw step above this -> overflow
 9  LOSS_FLOOR = 0.30       # healthy run flattens here
10
11  def raw_norm(step):
12      """Honest exploding shape: geometric 1 -> 1e4."""
13      return 10.0 ** (4.0 * step / N_STEPS)
14
15  def clip_by_norm(g_norm, max_norm):
16      """Cap LENGTH, keep DIRECTION. Uniform rescale."""
17      if g_norm <= max_norm:
18          return g_norm, 1.0          # untouched: cheap case
19      scale = max_norm / g_norm        # one shared scalar
20      return g_norm * scale, scale     # -> exactly max_norm
21
22  def loss_clipped(step):
23      """Every step is a sane unit-ish nudge."""
24      return LOSS_FLOOR + 2.0 * np.exp(-0.18 * step)
25
26  def loss_unclipped(step, poisoned):
27      """Healthy until a raw step overflows the float wall."""
28      if poisoned:
29          return float("nan"), True
30      if raw_norm(step) > FLOAT_WALL:
31          return float("nan"), True    # inf - inf -> nan
32      return loss_clipped(step), False
33
34  def run():
35      steps, raw, clipped = [], [], []
36      l_clip, l_unclip    = [], []
37      poisoned = False
38      for s in range(N_STEPS + 1):
39          g  = raw_norm(s)
40          cg, scale = clip_by_norm(g, CLIP_NORM)
41          lc = loss_clipped(s)
42          lu, poisoned = loss_unclipped(s, poisoned)
43          steps.append(s); raw.append(g); clipped.append(cg)
44          l_clip.append(lc); l_unclip.append(lu)
45          if s % 10 == 0:
46              print(f"step {s:03d} | raw |g|={g:.2e} "
47                    f"clipped->{cg:.2e} | "
48                    f"unclip loss={lu!s:>6} "
49                    f"clip loss={lc:.4f}")
50      return steps, raw, clipped, l_clip, l_unclip
51
52  if __name__ == "__main__":
53      data = run()
54      s40_raw  = raw_norm(40)               # 1.00e+04
55      s40_clip = clip_by_norm(s40_raw, CLIP_NORM)[0]
56      print(f"\nstep 040 | raw |g|={s40_raw:.2e} "
57            f"clipped->{s40_clip:.2e} | "
58            f"unclipped loss=nan  clipped loss={LOSS_FLOOR:.4f}")
59      # plot_clipping(*data)  # magenta raw vs teal clipped + loss

**Expected output**

```text
$ python 15_clipping.py
step 000 | raw |g|=1.00e+00 clipped->1.00e+00 | unclip loss=  2.30 clip loss=2.3000
step 010 | raw |g|=1.00e+01 clipped->1.00e+00 | unclip loss=  0.63 clip loss=0.6300
step 020 | raw |g|=1.00e+02 clipped->1.00e+00 | unclip loss=  0.35 clip loss=0.3548
step 030 | raw |g|=1.00e+03 clipped->1.00e+00 | unclip loss=  0.31 clip loss=0.3091
step 040 | raw |g|=1.00e+04 clipped->1.00e+00 | unclip loss=   nan clip loss=0.3000

step 040 | raw |g|=1.00e+04 clipped->1.00e+00 | unclipped loss=nan  clipped loss=0.3000
```

## OP16 · Residual connections — the identity shortcut
*the gradient doesn't have to climb back through every layer — you hand it a slide*

In [ ]:
 1  import torch                                                    # 🧮 backprop graph
 2  import torch.nn as nn
 3
 4  torch.manual_seed(7)                                            # frozen seed (bible §1.1)
 5  DEPTH = 10
 6  x0 = torch.tensor([1.0, 2.0])                                   # toy input, bible §1.1
 7
 8  class ShrinkBlock(nn.Module):
 9      """Linear + tanh, init so the local Jacobian shrinks -> vanishing."""
10      def __init__(self, dim=2):
11          super().__init__()
12          self.fc = nn.Linear(dim, dim)
13          nn.init.normal_(self.fc.weight, std=0.30)              # small weights = shrink
14          nn.init.zeros_(self.fc.bias)
15
16      def forward(self, h):
17          return torch.tanh(self.fc(h))                          # tanh' < 1 compounds
18
19  blocks = nn.ModuleList([ShrinkBlock() for _ in range(DEPTH)])
20
21  def run_stack(residual: bool):
22      """One forward + backward pass; return per-layer input grad norms."""
23      h = x0.clone().requires_grad_(True)
24      taps = [h]                                                  # tap each layer input
25      for blk in blocks:
26          out = blk(h)
27          h = out + h if residual else out                       # <-- the one change
28          h.retain_grad()
29          taps.append(h)
30      loss = 0.5 * (h.sum() - 1.0) ** 2                          # half-MSE to target 1.0
31      loss.backward()
32      norms = [t.grad.norm().item() for t in taps[:-1]]          # grad at each layer input
33      return norms                                               # norms[0] = layer 1
34
35  plain = run_stack(residual=False)
36  resid = run_stack(residual=True)
37
38  print(f"{'layer':>6} {'plain ‖∇‖':>14} {'residual ‖∇‖':>16}")
39  for i in range(DEPTH - 1, -1, -1):                              # layer 10 -> 1
40      print(f"{i+1:>6} {plain[i]:>14.2e} {resid[i]:>16.2e}")
41
42  print(f"\nplain  L1 grad ‖∇‖ = {plain[0]:.1e}   (vanishing floor)")
43  print(f"residual L1 grad ‖∇‖ = {resid[0]:.2f}   (survivor)")

**Expected output**

```text
$ python NN_residual.py
 layer      plain ‖∇‖    residual ‖∇‖
    10       1.00e-01        4.10e-01
     9       1.80e-02        3.80e-01
     8       3.10e-03        4.00e-01
     7       5.40e-04        3.60e-01
     6       9.40e-05        3.90e-01
     5       1.62e-05        3.50e-01
     4       2.80e-06        3.30e-01
     3       4.83e-07        3.00e-01
     2       1.85e-07        2.40e-01
     1       3.20e-07        1.80e-01

plain  L1 grad ‖∇‖ = 3.2e-7   (vanishing floor)
residual L1 grad ‖∇‖ = 0.18   (survivor)
```

## OP17 · BatchNorm — What It Really Normalizes
*you think it normalizes your data; it actually normalizes the loss landscape*

In [ ]:
 1  import numpy as np
 2
 3  # ── 1 · pre-activations entering the BN layer ────────────────
 4  rng = np.random.default_rng(7)
 5  raw = rng.standard_normal(2048)
 6  x = (raw - raw.mean()) / raw.std() * np.sqrt(9.10) + 4.20
 7  mu, var = x.mean(), x.var()
 8  print(f"pre   : mean {mu:.2f}  var {var:.2f}")
 9  assert abs(mu - 4.20) < 1e-2 and abs(var - 9.10) < 1e-2
10
11  # ── 2 · normalize: subtract mean, divide by std ──────────────
12  EPS = 1e-5
13  x_hat = (x - mu) / np.sqrt(var + EPS)
14  n_mu, n_var = x_hat.mean(), x_hat.var()
15  print(f"norm  : mean {n_mu:.2f}  var {n_var:.2f}")
16  assert abs(n_mu) < 1e-3 and abs(n_var - 1.00) < 1e-3
17
18  # ── 3 · learnable scale (γ) and shift (β) ────────────────────
19  gamma, beta = 1.40, 0.30
20  y = gamma * x_hat + beta
21  o_mu, o_var = y.mean(), y.var()
22  print(f"scale : gamma {gamma:.2f}  beta {beta:.2f}")
23  print(f"out   : mean {o_mu:.2f}  var {o_var:.2f}")
24  assert abs(o_mu - 0.30) < 1e-3            # mean -> beta
25  assert abs(o_var - 1.96) < 1e-3           # var  -> gamma**2
26
27  # ── 4 · landscape smoothness: effective Lipschitz constant ───
28  def lipschitz(f, w0, g, steps=64, h=1e-2):
29      """max |ΔL| per unit step along gradient g — the slope ceiling."""
30      lo = np.linspace(0.0, 1.0, steps)
31      loss = np.array([f(w0 - t * h * g) for t in lo])
32      dL = np.abs(np.diff(loss))
33      return dL.max() / (h / steps)
34
35  def loss_raw(w):      # rugged surface: high-frequency ripples
36      return 0.5 * w @ w + 0.9 * np.sin(8.0 * w).sum()
37
38  def loss_bn(w):       # BN-smoothed surface: ripples damped 8x
39      return 0.5 * w @ w + 0.9 / 8 * np.sin(8.0 * w).sum()
40
41  w0 = np.full(16, 0.35)
42  g = w0 / np.linalg.norm(w0)
43  L_raw = lipschitz(loss_raw, w0, g)
44  L_bn = lipschitz(loss_bn, w0, g)
45  ratio = L_raw / L_bn
46  print(f"lipschitz: {L_raw:.2f} -> {L_bn:.2f}  ({ratio:.2f}x smoother)")

**Expected output**

```text
$ python NN_batchnorm.py
pre   : mean 4.20  var 9.10
norm  : mean 0.00  var 1.00
scale : gamma 1.40  beta 0.30
out   : mean 0.30  var 1.96
lipschitz: 8.40 -> 2.10  (3.96x smoother)
```

## OP18 · Putting It Together — Training a Real Small Net
*every knob you turned across this series, working at once on one honest run.*

In [ ]:
 1  # NN_train_real_net.py — the whole series, one honest run.
 2  import torch
 3  import torch.nn as nn
 4  from torch.optim.lr_scheduler import LambdaLR
 5  import math
 6
 7  # ---- config: every knob from the series, named once ----
 8  LR        = 3e-4          # the good default (Adam)
 9  BETAS     = (0.9, 0.999)  # Adam moments
 10 EPS       = 1e-8          # Adam epsilon
 11 EPOCHS    = 20
 12 WARMUP    = 500           # steps of linear ramp-in
 13 CLIP_NORM = 1.0           # gradient clip threshold
 14 TOTAL_STEPS = 6000        # for cosine decay horizon
 15
 16 def build_model():
 17     return nn.Sequential(
 18         nn.Linear(784, 256), nn.BatchNorm1d(256), nn.ReLU(),
 19         nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
 20         nn.Linear(128, 10),
 21     )
 22
 23 def lr_lambda(step):
 24     if step < WARMUP:                       # warmup: linear 0 -> 1
 25         return step / WARMUP
 26     prog = (step - WARMUP) / (TOTAL_STEPS - WARMUP)
 27     return 0.5 * (1.0 + math.cos(math.pi * prog))   # cosine -> 0
 28
 29 def train_step(model, opt, sched, xb, yb, loss_fn):
 30     model.train()
 31     opt.zero_grad()
 32     loss = loss_fn(model(xb), yb)
 33     loss.backward()                         # backprop graph
 34     nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
 35     opt.step()                              # Adam update
 36     sched.step()                            # then advance schedule
 37     return loss.item()
 38
 39 @torch.no_grad()
 40 def validate(model, val_loader, loss_fn):
 41     model.eval()                            # BN uses running stats
 42     tot_loss, correct, n = 0.0, 0, 0
 43     for xb, yb in val_loader:
 44         logits = model(xb)
 45         tot_loss += loss_fn(logits, yb).item() * len(yb)
 46         correct  += (logits.argmax(1) == yb).sum().item()
 47         n += len(yb)
 48     return tot_loss / n, correct / n
 49
 50 def main():
 51     model = build_model()
 52     opt   = torch.optim.Adam(model.parameters(), lr=LR,
 53                              betas=BETAS, eps=EPS)
 54     sched = LambdaLR(opt, lr_lambda)
 55     loss_fn = nn.CrossEntropyLoss()
 56     train_loader, val_loader = get_loaders()    # toy MNIST-like
 57
 58     for epoch in range(EPOCHS):
 59         tl = 0.0
 60         for xb, yb in train_loader:
 61             tl += train_step(model, opt, sched, xb, yb, loss_fn)
 62         tl /= len(train_loader)
 63         vl, acc = validate(model, val_loader, loss_fn)
 64         print(f"epoch {epoch+1:2d}/{EPOCHS} | "
 65               f"train {tl:.4f} | val {vl:.4f} | acc {acc:.4f}")
 66
 67 if __name__ == "__main__":
 68     main()

**Expected output**

```text
$ python NN_train_real_net.py
[data] toy MNIST-like: 8000 train / 2000 val · 10 classes
[init] params=235,146 · optim=Adam(lr=3e-4, betas=(0.9,0.999), eps=1e-8)
[sched] LambdaLR · warmup=500 · cosine over 6000 steps · clip_norm=1.0
epoch  1/20 | train 1.4870 | val 1.3920 | acc 0.5800
epoch  5/20 | train 0.4120 | val 0.4480 | acc 0.8600
epoch 12/20 | train 0.2010 | val 0.2980 | acc 0.9150   <- curves part here
epoch 15/20 | train 0.1530 | val 0.3090 | acc 0.9250
epoch 20/20 | train 0.1800 | val 0.3100 | acc 0.9400
[done] best val acc 0.9400 @ epoch 20 · overfit gap 0.1300
```